In [2]:
import os
import nibabel as nib
import numpy as np
import cv2
from tqdm import tqdm

# ===============================
# PATHS (ABSOLUTE - VERIFIED)
# ===============================
RAW_DATA_PATH = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
OUTPUT_IMG_PATH = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\images"
OUTPUT_LBL_PATH = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\labels"

# ===============================
# CREATE OUTPUT DIRECTORIES
# ===============================
os.makedirs(OUTPUT_IMG_PATH, exist_ok=True)
os.makedirs(OUTPUT_LBL_PATH, exist_ok=True)

# ===============================
# BASIC CHECKS
# ===============================
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f"RAW_DATA_PATH not found: {RAW_DATA_PATH}")

print("Dataset found ✅")
print("Images will be saved to:", OUTPUT_IMG_PATH)
print("Labels will be saved to:", OUTPUT_LBL_PATH)
print("Current working directory:", os.getcwd())

# ===============================
# PARAMETERS
# ===============================
IMG_SIZE = 224
MAX_PATIENTS = 50  # start small for fast experimentation

# ===============================
# NORMALIZATION (SAFE)
# ===============================
def normalize(img):
    max_val = np.max(img)
    if max_val == 0:
        return img
    img = img - np.min(img)
    img = img / max_val
    return img

# ===============================
# LOAD PATIENTS
# ===============================
patient_folders = sorted(os.listdir(RAW_DATA_PATH))
print(f"Total patients found: {len(patient_folders)}")

slice_count = 0

# ===============================
# PROCESS PATIENTS
# ===============================
for patient in tqdm(patient_folders[:MAX_PATIENTS], desc="Processing patients"):
    patient_path = os.path.join(RAW_DATA_PATH, patient)

    # Locate files
    flair_files = [f for f in os.listdir(patient_path) if "t2w" in f.lower()]
    seg_files = [f for f in os.listdir(patient_path) if "seg" in f.lower()]

    if len(flair_files) == 0 or len(seg_files) == 0:
        print(f"Skipping {patient} (missing FLAIR or SEG)")
        continue

    flair_path = os.path.join(patient_path, flair_files[0])
    seg_path = os.path.join(patient_path, seg_files[0])

    # Load volumes
    flair = nib.load(flair_path).get_fdata()
    seg = nib.load(seg_path).get_fdata()

    # ===============================
    # SLICE EXTRACTION
    # ===============================
    for i in range(flair.shape[2]):
        flair_slice = flair[:, :, i]
        seg_slice = seg[:, :, i]

        # Skip empty slices
        if np.max(flair_slice) == 0:
            continue

        # Normalize & resize
        flair_slice = normalize(flair_slice)
        flair_slice = cv2.resize(flair_slice, (IMG_SIZE, IMG_SIZE))

        # Label: tumor (1) or non-tumor (0)
        label = 1 if np.max(seg_slice) > 0 else 0

        # Filenames
        img_name = f"{patient}_slice_{i}.png"
        lbl_name = f"{patient}_slice_{i}.txt"

        # Save image
        cv2.imwrite(
            os.path.join(OUTPUT_IMG_PATH, img_name),
            (flair_slice * 255).astype(np.uint8)
        )

        # Save label
        with open(os.path.join(OUTPUT_LBL_PATH, lbl_name), "w") as f:
            f.write(str(label))

        slice_count += 1

print("\n===============================")
print("Preprocessing completed ✅")
print("Total slices saved:", slice_count)
print("===============================")


Dataset found ✅
Images will be saved to: C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\images
Labels will be saved to: C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\labels
Current working directory: c:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks
Total patients found: 1251


Processing patients:   0%|          | 0/50 [00:00<?, ?it/s]

Processing patients: 100%|██████████| 50/50 [01:01<00:00,  1.24s/it]


Preprocessing completed ✅
Total slices saved: 6939
